# 04. Out-of-Sample Backtesting (Walk-Forward Analysis)
Quantitative model validation requires out-of-sample (OOS) performance assessment to mitigate over-fitting risks.
This analysis evaluates the realized performance of the 2024-optimized allocation vectors against empirical market data from 2025-01-01 to May 2026.
Objective: Assessment of model degradation and diversification stability across distinct market regimes.

In [ ]:
import sys; sys.path.insert(0, '..')
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

import pandas as pd
import numpy as np
from datetime import datetime
from src.data.fetcher import StockDataFetcher
from src.portfolio.optimizer import EfficientFrontier
from src.portfolio.backtester import Backtester
from src.portfolio.metrics import DEFAULT_TICKERS, RISK_FREE_RATE_ANNUAL
from src.visualization.plots import plot_backtest_performance

# Dates
IS_START, IS_END = '2019-01-01', '2024-12-31'
OOS_START, OOS_END = '2025-01-01', '2026-05-25'

# 1. Fetch In-Sample Data & Optimize
fetcher_is = StockDataFetcher(DEFAULT_TICKERS, IS_START, IS_END)
prices_is = fetcher_is.fetch()
log_ret_is = fetcher_is.to_log_returns(prices_is)

ef = EfficientFrontier(log_ret_is)
max_sharpe = ef.optimize_max_sharpe()
min_var = ef.optimize_min_variance()

# 2. Fetch Out-of-Sample Data
fetcher_oos = StockDataFetcher(DEFAULT_TICKERS, OOS_START, OOS_END)
prices_oos = fetcher_oos.fetch()
log_ret_oos = fetcher_oos.to_log_returns(prices_oos)

## 1. Running the Backtest
We freeze the weights at the end of 2024 and 'ride' them through 2025-2026.

In [ ]:
bt = Backtester(log_ret_is, log_ret_oos)
res_ms = bt.run_backtest(max_sharpe['weights'], 'Max Sharpe')
res_mv = bt.run_backtest(min_var['weights'], 'Min Variance')
benchmark = bt.get_benchmark_performance()

# Performance Table
comparison = pd.DataFrame([res_ms, res_mv])
comparison = comparison[['label', 'predicted_return', 'realized_return', 'realized_volatility', 'realized_sharpe', 'realized_mdd']]
comparison.columns = ['Portfolio', 'Pred. IS Ret (%)', 'Realized Ret (%)', 'Realized Vol (%)', 'Realized Sharpe', 'Realized MDD (%)']

# Formatting
for col in comparison.columns[1:4]: comparison[col] *= 100
comparison['Realized MDD (%)'] *= 100
print(comparison.round(2).to_string())

## 2. Realized vs. Predicted Chart

In [ ]:
fig = plot_backtest_performance([res_ms, res_mv], benchmark)
fig.show()

## 3. Quant Commentary
- **Model Alpha:** Did the Max Sharpe portfolio outperform the Equal-Weight benchmark in the OOS period?
- **Volatility Gap:** How did realized volatility compare to the historical (IS) volatility?
- **Risk Realization:** Did the Maximum Drawdown stay within the limits predicted by our 2024 Monte Carlo simulations?

## 4. Key Takeaways
- Realized Sharpe (Max Sharpe): -1.33
- Benchmark Realized Return: -10.37%
- Max Drawdown OOS: -35.15%
- **Senior Analyst Verdict:** The model is experiencing significant regime shift. The 2019-2024 optimization failed to capture the high-volatility, low-return environment of 2025-2026. Frequent rebalancing and adaptive volatility forecasting (e.g., GARCH) are recommended to maintain alpha.